In [ ]:
import sys
import json
from tqdm import tqdm
from openai import OpenAI 
import os
from gigachat import GigaChat 
import joblib
from gigachat.models import Chat, Messages 
from functools import reduce
from typing import Dict
import gc

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

from src.pipelines.memorize import MemPipelineConfig, MemPipeline, LLMExtractorConfig, LLMUpdatorConfig
from src.kg_model import KnowledgeGraphModel, EmbeddingsModelConfig, GraphModelConfig, EmbedderModelConfig
from src.db_drivers.graph_driver import GraphDBConnectionConfig, GraphDriverConfig
from src.db_drivers.vector_driver import VectorDBConnectionConfig, VectorDriverConfig

# gigachat key
#GIGACHAT_CREDS = 'OWUwOGUzOWEtMjJiNi00YmMxLThmMmItNzMwNjM2MTI2YmYxOjg2ODdiOTVhLTZkNDctNGFjOC1iMmViLTEyNDA5MmFiN2Q5Mw=='
# openai key
#API_KEY = "'sk-861mINAavom2SSBqgrI82D4thMOfqT37knCof2o0H0T3BlbkFJ2gdVXJuVjNesNNP2aeUwPoBpZP3a3R1gn1kqv97CsA'"

DATASET_PATH = '../data/Augment_DiaASQ.json'

SAVE_EXTRACTED_TRIPLETS_FILE = "extracted_triplets.json"
GRAPH_DRIVER_CONFIG_FILE = "graph_config"
EMBEDDINGS_DRIVER_CONFIG_FILE = "embeddings_config"
MEM_PIPELINE_CONFIG_FILE = "mem_pipeline_config"

PARAMS = {
    
}
gc.collect()

In [ ]:
BASE_PATH = "../../data/knowledge_graphs/"
DATASET_PATH = BASE_PATH + "diaasq/"
KG_PATH = DATASET_PATH + "/gigachat_full"

In [ ]:
VECTORIZED_DB_PATH = KG_PATH + "/embeddings_part"
GRAPH_DB_PATH = KG_PATH + "/graph_part"
EMBEDDER_MODEL_PATH = '../../models/intfloat/multilingual-e5-small'

kg_model = KnowledgeGraphModel(
    graph_config=GraphModelConfig(
        driver_config=GraphDriverConfig(db_vendor='neo4j', 
            db_config=GraphDBConnectionConfig(
                uri="bolt://localhost:7687", 
                params={'user': "neo4j", 'pwd': 'password'}, 
                need_to_clear=True))),
    embd_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_vendor='chroma', 
            db_config=VectorDBConnectionConfig(path=VECTORIZED_DB_PATH, 
                db_info={'db': 'personalai_db', 'table': "vectorized_nodes"}, need_to_clear=True),
        tripletsdb_driver_config=VectorDriverConfig(db_vendor='chroma', 
            db_config=VectorDBConnectionConfig(path=VECTORIZED_DB_PATH, 
                db_info={'db': 'personalai_db', 'table': "vectorized_triplets"}, need_to_clear=True)),
        embedder_config=EmbedderModelConfig(
           model_name_or_path=EMBEDDER_MODEL_PATH))))

mem_config = MemPipelineConfig(
    extractor_config=LLMExtractorConfig(),
    updator_config=LLMUpdatorConfig(
        delete_obsolete_info=False))

mem_pipeline = MemPipeline(kg_model, mem_config)

In [8]:
with open(DATASET_PATH, 'r', encoding='utf-8') as fd:
    data = json.loads(fd.read())

raw_texts = list(map(lambda v: v['text_dialog'], data['data']))
raw_time = list(map(lambda v: v['time'].split(',')[0], data['data']))
print(len(raw_texts), len(raw_time))

In [ ]:
saved_triplets = []
for text, time in zip(raw_texts, raw_time):
    extracted_triplets, _ = mem_pipeline.remember(text, {'time': time})
    saved_triplets.append(extracted_triplets)

In [ ]:
joblib.dump(extracted_triplets, KG_PATH + ''SAVE_EXTRACTED_TRIPLETS_FILE)